# Exploratory Data Analysis: Student Depression Dataset

This notebook explores the student depression dataset to understand patterns, distributions, and relationships between features.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Load and Inspect Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/student_depression_dataset.csv')

# Clean string columns - remove extra quotes from values
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.strip("'")

print(f"Dataset Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
df.head()

In [ ]:
# Data types and info
print("Data Types:")
print("-" * 40)
print(df.dtypes)
print("\n" + "=" * 40)
df.info()

In [ ]:
# Statistical summary
df.describe(include='all').T

## 2. Missing Values Analysis

In [ ]:
# Check for missing values
missing_data = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing Count'] > 0].sort_values('Missing Percentage', ascending=False)

if len(missing_data) == 0:
    print("✓ No missing values in the dataset!")
else:
    print("Missing Values Summary:")
    display(missing_data)
    
    # Visualize missing values
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Heatmap')
    plt.tight_layout()
    plt.show()

## 3. Target Variable Analysis (Depression)

In [ ]:
# Depression distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
depression_counts = df['Depression'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(depression_counts.index, depression_counts.values, color=colors)
axes[0].set_xlabel('Depression Status')
axes[0].set_ylabel('Count')
axes[0].set_title('Depression Distribution')
for i, v in enumerate(depression_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(depression_counts.values, labels=['No Depression', 'Depression'], 
            autopct='%1.1f%%', colors=colors, explode=(0, 0.05),
            shadow=True, startangle=90)
axes[1].set_title('Depression Percentage Distribution')

plt.tight_layout()
plt.show()

print(f"\nClass Distribution:")
print(f"  No Depression: {depression_counts[0]:,} ({depression_counts[0]/len(df)*100:.1f}%)")
print(f"  Depression: {depression_counts[1]:,} ({depression_counts[1]/len(df)*100:.1f}%)")

## 4. Numerical Features Distribution

In [ ]:
# Identify numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [col for col in numerical_cols if col not in ['id', 'Depression']]

print(f"Numerical columns: {numerical_cols}")

# Distribution plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols[:9]):
    sns.histplot(data=df, x=col, hue='Depression', kde=True, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} Distribution by Depression')
    axes[i].legend(['No Depression', 'Depression'])

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols[:9]):
    sns.boxplot(data=df, x='Depression', y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} by Depression Status')
    axes[i].set_xticklabels(['No Depression', 'Depression'])

plt.tight_layout()
plt.show()

## 5. Categorical Features Analysis

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Show unique values for each categorical column
for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(f"  Values: {df[col].unique()[:10]}{'...' if df[col].nunique() > 10 else ''}")

In [ ]:
# Gender distribution by Depression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender count
gender_depression = pd.crosstab(df['Gender'], df['Depression'])
gender_depression.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Gender Distribution by Depression')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(['No Depression', 'Depression'])
axes[0].tick_params(axis='x', rotation=0)

# Depression rate by Gender
depression_rate = df.groupby('Gender')['Depression'].mean() * 100
depression_rate.plot(kind='bar', ax=axes[1], color=['#3498db', '#9b59b6'])
axes[1].set_title('Depression Rate by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Depression Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(depression_rate.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Sleep Duration and Dietary Habits analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sleep Duration
sleep_order = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours', 'Others']
sleep_data = df[df['Sleep Duration'].isin(sleep_order)]
sleep_depression = pd.crosstab(pd.Categorical(sleep_data['Sleep Duration'], categories=sleep_order, ordered=True), 
                               sleep_data['Depression'])
sleep_depression.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Sleep Duration by Depression')
axes[0].set_xlabel('Sleep Duration')
axes[0].set_ylabel('Count')
axes[0].legend(['No Depression', 'Depression'])
axes[0].tick_params(axis='x', rotation=45)

# Dietary Habits
diet_order = ['Unhealthy', 'Moderate', 'Healthy', 'Others']
diet_data = df[df['Dietary Habits'].isin(diet_order)]
diet_depression = pd.crosstab(pd.Categorical(diet_data['Dietary Habits'], categories=diet_order, ordered=True), 
                              diet_data['Depression'])
diet_depression.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Dietary Habits by Depression')
axes[1].set_xlabel('Dietary Habits')
axes[1].set_ylabel('Count')
axes[1].legend(['No Depression', 'Depression'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Suicidal Thoughts analysis - Critical feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Suicidal thoughts distribution
suicidal_col = 'Have you ever had suicidal thoughts ?'
suicidal_depression = pd.crosstab(df[suicidal_col], df['Depression'])
suicidal_depression.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Suicidal Thoughts by Depression Status')
axes[0].set_xlabel('Have you ever had suicidal thoughts?')
axes[0].set_ylabel('Count')
axes[0].legend(['No Depression', 'Depression'])
axes[0].tick_params(axis='x', rotation=0)

# Depression rate by suicidal thoughts
suicidal_rate = df.groupby(suicidal_col)['Depression'].mean() * 100
suicidal_rate.plot(kind='bar', ax=axes[1], color=['#27ae60', '#c0392b'])
axes[1].set_title('Depression Rate by Suicidal Thoughts History')
axes[1].set_xlabel('Have you ever had suicidal thoughts?')
axes[1].set_ylabel('Depression Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(suicidal_rate.values):
    axes[1].text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Prepare data for correlation (encode categorical variables)
df_encoded = df.copy()

# Encode categorical columns for correlation
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Drop id column if present
if 'id' in df_encoded.columns:
    df_encoded = df_encoded.drop('id', axis=1)

# Correlation matrix
plt.figure(figsize=(16, 12))
correlation_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with Depression (Target Variable)
depression_corr = correlation_matrix['Depression'].drop('Depression').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in depression_corr.values]
depression_corr.plot(kind='barh', color=colors)
plt.xlabel('Correlation with Depression')
plt.title('Feature Correlation with Depression')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop Positive Correlations with Depression:")
print(depression_corr.head(5))
print("\nTop Negative Correlations with Depression:")
print(depression_corr.tail(5))

## 7. Interactive Visualizations with Plotly

In [ ]:
# Interactive Age vs Academic Pressure scatter plot
fig = px.scatter(df, x='Age', y='Academic Pressure', color='Depression',
                 color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
                 title='Age vs Academic Pressure by Depression Status',
                 labels={'Depression': 'Depression Status'},
                 hover_data=['Gender', 'CGPA'])
fig.update_layout(legend_title_text='Depression')
fig.show()

In [ ]:
# Interactive sunburst chart
fig = px.sunburst(df, path=['Gender', 'Sleep Duration', 'Depression'], 
                  title='Hierarchical View: Gender → Sleep Duration → Depression',
                  color='Depression',
                  color_discrete_map={0: '#2ecc71', 1: '#e74c3c'})
fig.show()

## 8. Age Distribution Analysis

In [ ]:
# Age group analysis
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 20, 25, 30, 40, 100], 
                         labels=['<20', '20-25', '26-30', '31-40', '40+'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age group distribution
age_depression = pd.crosstab(df['Age_Group'], df['Depression'])
age_depression.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Age Group Distribution by Depression')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')
axes[0].legend(['No Depression', 'Depression'])
axes[0].tick_params(axis='x', rotation=0)

# Depression rate by age group
age_rate = df.groupby('Age_Group')['Depression'].mean() * 100
age_rate.plot(kind='bar', ax=axes[1], color='#9b59b6')
axes[1].set_title('Depression Rate by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Depression Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(age_rate.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Clean up temporary column
df = df.drop('Age_Group', axis=1)

## 9. Key Insights Summary

In [ ]:
# Generate summary statistics
print("=" * 60)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   • Total samples: {len(df):,}")
print(f"   • Features: {len(df.columns) - 1}")
print(f"   • Missing values: {df.isnull().sum().sum()}")

depression_rate = df['Depression'].mean() * 100
print(f"\n🎯 TARGET VARIABLE (Depression):")
print(f"   • Depression rate: {depression_rate:.1f}%")
print(f"   • Class imbalance: {'Yes (moderate)' if abs(50 - depression_rate) > 10 else 'Balanced'}")

print(f"\n🔍 TOP CORRELATED FEATURES WITH DEPRESSION:")
top_features = depression_corr.head(5)
for feat, corr in top_features.items():
    print(f"   • {feat}: {corr:.3f}")

print(f"\n💡 KEY OBSERVATIONS:")
print("   • Suicidal thoughts history is strongly correlated with depression")
print("   • Academic pressure shows significant positive correlation")
print("   • Financial stress is a notable risk factor")
print("   • CGPA has notable correlation with depression status")
print("   • Sleep duration and dietary habits show relationships with mental health")

print("\n" + "=" * 60)
print("EDA COMPLETE - Ready for Model Training!")
print("=" * 60)